<a href="https://colab.research.google.com/github/MoharanaSudhanshu/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MoharanaSudhanshu/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of Analysis

One row represents one webpage observed during a specific historical time window. Each row contains anonymized search performance metrics such as impressions, clicks, click-through rate (CTR), average position, and content age.

### Time Window

This project uses historical search performance data collected before prediction. The model uses past observations to determine whether a webpage is likely to experience declining search performance and should be considered for content refresh.

In [2]:
!git clone https://github.com/MoharanaSudhanshu/flyrank-ml-internship.git

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 133, done.
remote: Counting objects: 100% (133/133), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 133 (delta 50), reused 91 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (133/133), 1.86 MiB | 10.98 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [3]:
%cd flyrank-ml-internship

/content/flyrank-ml-internship


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Rows: 30000
Columns: 44


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## Features

The model will use:

- impressions_90d
- ctr
- avg_position
- content_age_days
- trend_direction

These variables contain historical information available before prediction.

---

## Label (Target)

The target is whether a webpage is likely to experience declining search performance and should be prioritized for content refresh.

---

## Context Fields

Context information includes:

- observation month
- page metadata
- content category

These help interpret the data but are not necessarily predictive features.

---

## Excluded Fields

I deliberately exclude:

- Page URLs
- Client identifiers
- Future clicks
- Future impressions
- Future CTR
- Any manually created labels from the future

These either identify clients or would introduce data leakage.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rows:", len(df))
print("Columns:", len(df.columns))

Rows: 30000
Columns: 44


In [7]:
df.isnull().sum()

,0
content_id,0
client_id,0
search_volume,2468
competition,2468
competition_level,2610
cpc,2468
content_type,0
main_intent,2374
word_count,7699
char_count,7699


In [8]:
print("Average CTR:", round(df["ctr"].mean(),3))
print("Average Position:", round(df["avg_position"].mean(),2))
print("Average Content Age:", round(df["content_age_days"].mean(),2))

Average CTR: 0.511
Average Position: 16.34
Average Content Age: 256.17


In [9]:
df.describe()

,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,sessions_prev_30d,content_age_days,age_tier_order,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,trend_pct
count,27532.000000,27532.000000,27532.000000,22301.000000,22301.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.00000,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000,29875.000000,30000.000000,26612.000000
mean,158.882391,0.146954,0.485342,3107.760325,20665.277835,5200.366300,16.097333,49.942467,37.066633,35.937700,...,10.283000,256.16780,4.786533,46.098300,0.510733,16.34238,2.534520,18.212921,0.768196,-4.785969
std,1518.270825,0.285241,2.101560,1452.382598,10115.344042,16838.019547,75.076958,152.101430,107.069131,103.748185,...,42.578003,132.70793,0.790392,42.078709,3.279162,15.21679,8.310096,29.472768,7.429454,473.861780
min,0.000000,0.000000,0.000000,8.000000,40.000000,1.000000,0.000000,0.000000,1.000000,1.000000,...,0.000000,90.00000,3.000000,1.000000,0.000000,0.00000,0.000000,0.000000,0.000000,-100.000000
25%,0.000000,0.000000,0.000000,2413.000000,15644.000000,81.000000,0.000000,2.000000,2.000000,2.000000,...,1.000000,132.00000,4.000000,20.000000,0.000000,6.20000,0.000000,0.000000,0.000000,-62.600000
50%,10.000000,0.000000,0.000000,2877.000000,19116.000000,731.000000,1.000000,8.000000,7.000000,7.000000,...,2.000000,236.00000,5.000000,20.000000,0.070000,10.80000,0.000000,5.000000,0.000000,-33.500000
75%,20.000000,0.130000,0.000000,3666.000000,24011.000000,3615.250000,7.000000,33.000000,27.000000,27.000000,...,7.000000,333.00000,5.000000,104.000000,0.290000,22.30000,1.350000,23.530000,0.000000,0.000000
max,74000.000000,1.000000,100.360000,9546.000000,111158.000000,517715.000000,4178.000000,5998.000000,4345.000000,4913.000000,...,4247.000000,564.00000,6.000000,373.000000,100.000000,245.00000,100.000000,300.000000,300.000000,44900.000000


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Data Limitations

This dataset contains historical observations and cannot prove causal relationships.

The data does not explain Google's ranking algorithm or guarantee that refreshing a page will improve rankings.

The dataset is anonymized, so client identities and webpage URLs cannot be recovered.

Historical search behaviour may change over time, meaning patterns learned from past data may not generalize perfectly to future search trends.

The model should be used only as decision support for identifying pages that may require review rather than making automatic business decisions.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [12]:
df.describe(include="all")

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
count,30000,30000,27532.000000,27532.000000,27390,27532.000000,30000,27626,22301.000000,22301.000000,...,22301,30000.000000,30000.00000,30000.000000,29875.000000,30000.000000,30000,30000,30000,26612.000000
unique,30000,32,NaN,NaN,3,NaN,3,4,NaN,NaN,...,4,NaN,NaN,NaN,NaN,NaN,4,5,5,NaN
top,content_6880eb215048,client_19581e27de,NaN,NaN,LOW,NaN,keyword article,informational,NaN,NaN,...,15000-25000,NaN,NaN,NaN,NaN,NaN,low,page_1,down,NaN
freq,1,7008,NaN,NaN,22896,NaN,27207,17235,NaN,NaN,...,12055,NaN,NaN,NaN,NaN,NaN,11248,11814,16262,NaN
mean,NaN,NaN,158.882391,0.146954,NaN,0.485342,NaN,NaN,3107.760325,20665.277835,...,NaN,0.510733,16.34238,2.534520,18.212921,0.768196,NaN,NaN,NaN,-4.785969
std,NaN,NaN,1518.270825,0.285241,NaN,2.101560,NaN,NaN,1452.382598,10115.344042,...,NaN,3.279162,15.21679,8.310096,29.472768,7.429454,NaN,NaN,NaN,473.861780
min,NaN,NaN,0.000000,0.000000,NaN,0.000000,NaN,NaN,8.000000,40.000000,...,NaN,0.000000,0.00000,0.000000,0.000000,0.000000,NaN,NaN,NaN,-100.000000
25%,NaN,NaN,0.000000,0.000000,NaN,0.000000,NaN,NaN,2413.000000,15644.000000,...,NaN,0.000000,6.20000,0.000000,0.000000,0.000000,NaN,NaN,NaN,-62.600000
50%,NaN,NaN,10.000000,0.000000,NaN,0.000000,NaN,NaN,2877.000000,19116.000000,...,NaN,0.070000,10.80000,0.000000,5.000000,0.000000,NaN,NaN,NaN,-33.500000
75%,NaN,NaN,20.000000,0.130000,NaN,0.000000,NaN,NaN,3666.000000,24011.000000,...,NaN,0.290000,22.30000,1.350000,23.530000,0.000000,NaN,NaN,NaN,0.000000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.